# Preparation of annotated 'variants' data

In [1]:
from var_utils import *

In [2]:
## Papers sourced via Claude search in ePMC, selected for having a variety of styles of genetic variant mentions
papers = ["PMC7334197", "PMC12713268", "PMC12465344", "PMC12859152", "PMC12874668", "PMC4560075", "PMC11354791", "PMC8254301"]
papers

['PMC7334197',
 'PMC12713268',
 'PMC12465344',
 'PMC12859152',
 'PMC12874668',
 'PMC4560075',
 'PMC11354791',
 'PMC8254301']

In [3]:
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
sys.path.insert(0, os.path.normpath(os.path.join(HERE, '../../../epmc-tools/europmc_dev_tool')))
from spacy_patterns import patterns

In [4]:
from transformers import AutoTokenizer
# Load the same tokenizer the pipeline uses
# _tokenizer = AutoTokenizer.from_pretrained("pruas/BENT-PubMedBERT-NER-Gene")
_tokenizer = AutoTokenizer.from_pretrained("OTAR3088/BioCODGE-GO")

def run_gene_ner_chunked(text: str, pipe, max_tokens: int = 400) -> list[dict]:
    """
    Chunk text at token boundaries (not char boundaries) to stay safely
    under the 512-token position embedding limit.
    """
    # Tokenize with char offset mapping, no special tokens
    enc = _tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False
    )
    token_ids   = enc["input_ids"]
    offset_map  = enc["offset_mapping"]   # (char_start, char_end) per token
    results = []
    for chunk_start in range(0, len(token_ids), max_tokens):
        chunk_offsets = offset_map[chunk_start : chunk_start + max_tokens]
        char_start    = chunk_offsets[0][0]
        char_end      = chunk_offsets[-1][1]
        chunk_text    = text[char_start:char_end]
        for ent in pipe(chunk_text):
            ent = dict(ent)
            ent["start"] += char_start   # remap to original text offsets
            ent["end"]   += char_start
            results.append(ent)
    return results


##### Collating all regex / NER annotations of articles, writing to LabelStudio json format

In [5]:
import json
import uuid
from transformers import pipeline

all_tasks = [] # All LabelStudio tasks (i.e. sections of text)

# BioCODGE-GO, labels multiple entity types, filtered here for GeneProtein
ner_gene = pipeline(
    "ner",
    model="OTAR3088/BioCODGE-GO",
    aggregation_strategy="simple"   # merges subword tokens into full spans
)

# 'patterns' imported here sourced from ePMC
label_list = ["refsnp", "gca"] # relevant patterns
var_patterns = [x for x in patterns if x['label'] in label_list]

for pmcid in papers:
    xml_out = get_epmc_full_text(pmcid=pmcid)
    if xml_out is None:
        continue
    parsed  = parse_epmc_xml(pmcid=pmcid, xml_text=xml_out)


    for sec in parsed.sections:
        ls_tasks = [] # tasks (sections) for paper 'pmcid'
        epmc_matches = []

        text = '\n'.join(sec['text'].split(sep='. ')) # prepare so as to not mess span numbers
        text = map_to_ascii(text) # catch non-ASCII chars

        for p in var_patterns:
            pattern = p['pattern']
            epmc_matches.extend([(m.group(), m.start(), m.end(), "ePMCVar") for m in re.finditer(pattern, text)])
        matches = [(m.group(), m.start(), m.end(), "HGVSVar") for m in HGVS.finditer(text)]
        genome = [(m.group(), m.start(), m.end(), "Refgenome") for m in GENOME_RE.finditer(text)]
        iscn = [(m.group(), m.start(), m.end(), "ISCNVar") for m in CYTOBAND.finditer(text)]
        
        # Gathering matches
        matches.extend(epmc_matches)
        matches.extend(genome)
        matches.extend(iscn)

        # Add geneprotein matches
        bert_results = run_gene_ner_chunked(text, ner_gene)
        # Keeping only 'GP' annotations
        bert_results = [(ent['word'], ent['start'], ent['end'], "GeneProtein") for ent in bert_results if ent.get('entity_group') == "GP"]


        # Catch star allele descriptions if present as 'Variant', otherwise keep as 'GeneProtein'
        star_checked = find_star_alleles(text, bert_results)
        matches.extend(star_checked)

        # Add to LabelStudio-compliant json
        for m in matches:
            ls_tasks.append({
                "value": {
                    "start": m[1],
                    "end":   m[2],
                    "text":  m[0],
                    "labels": [m[3]]
                },
                "id":        str(uuid.uuid4())[:10],
                "from_name": "label",
                "to_name":   "text",
                "type":      "labels",
                "origin":    "prediction"  # marks as pre-annotation, not human
            })

        all_tasks.append({
            "data": {
                "text":    text,
                "heading": sec["heading"] or "Body",
                "pmcid":   pmcid,
            },
            "annotations": [
                {
                    "result":       ls_tasks,
                    "was_cancelled": False,
                    "ground_truth": False,
                }
            ]
        })

output_path = f"./output/genevar_outputs/variants_ls.json"
with open(output_path, "w") as f:
    json.dump(all_tasks, f, indent=2)
print(f"Written {len(all_tasks)} tasks to {output_path}")

Device set to use mps:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/Users/withers/GitProjects/OTAR3088/ner-model/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Written 123 tasks to ./output/genevar_outputs/variants_ls.json


##### Some example annotation phrases for review

In [6]:
# # Note written case like 'BRAF V600E mutation' not covered
# Extra format examples
extras = [
    "Dummy example",
    "g.140453136A>T",
    "n.45G>C",
    "r.76a>u",
    "c.76_78del",
    "c.76_77insACG",
    "p.Arg213*",
    "p.Val600=",
    "p.V600E",
    "ENST00000288135.6:c.1799T>A",
    '( NM_001184.4 ):c.4405 A > G',
    'p.(Thr1469Ala)',
    '( NM_139076.3 ):c.727 C > G',
    'p.(Leu243Val)',
    '( NM_004448.4 ):c.3565G > C',
    'p.(Gly1189Arg)',
    '( NM_023110.3 ):c.386 A > C',
    'p.(Asp129Ala)',
    '( NM_007194.4 ):c.906 A > C',
    'p.(Glu302Asp)',
    'c.3306 C > G',
    '*1/*1  +  c.940   C  >  A '
]
print("\n-------MORE EXAMPLES--------")
for e in extras:
    m = HGVS.search(e)
    print(f"  {'OK' if m else 'X'}  {e!r:40s}  -> {m.group() if m else 'no match'}")


-------MORE EXAMPLES--------
  X  'Dummy example'                           -> no match
  OK  'g.140453136A>T'                          -> g.140453136A>T
  OK  'n.45G>C'                                 -> n.45G>C
  OK  'r.76a>u'                                 -> r.76a>u
  OK  'c.76_78del'                              -> c.76_78del
  OK  'c.76_77insACG'                           -> c.76_77insACG
  OK  'p.Arg213*'                               -> p.Arg213*
  OK  'p.Val600='                               -> p.Val600=
  OK  'p.V600E'                                 -> p.V600E
  OK  'ENST00000288135.6:c.1799T>A'             -> ENST00000288135.6:c.1799T>A
  OK  '( NM_001184.4 ):c.4405 A > G'            -> ( NM_001184.4 ):c.4405 A > G
  OK  'p.(Thr1469Ala)'                          -> p.(Thr1469Ala
  OK  '( NM_139076.3 ):c.727 C > G'             -> ( NM_139076.3 ):c.727 C > G
  OK  'p.(Leu243Val)'                           -> p.(Leu243Val
  OK  '( NM_004448.4 ):c.3565G > C'             -> (